Occupancy Data Cleaning.
Objective
This notebook converts irregular occupancy event logs into regular time series suitable for forecasting.
*AI was used in this notebook, primarily so that everything would stay oganised, I gave pseudo code, named all variables and personally checked data to give CoPilot the correct inputs, context and project scope. Each Subpart 1-18 was done one at a time, whilst I verified everything**
The main tasks are:

Load the occupancy event records
Validate timestamps, spaces and headcount values
Anonymise floor-space identifiers
Reconstruct occupancy as a step function over time
Aggregate occupancy using hourly mean, time-weighted mean and integral-based methods
Compare aggregation methods
Evaluate convergence as the time-bin width decreases
Save a processed hourly dataset for modelling
Environmental sensor data is excluded because the environmental sensor IDs cannot be reliably mapped to occupancy floor-space IDs.

Preliminary aggregation findings
The raw occupancy observations were converted into a regular 15-minute series using the unweighted arithmetic mean of the recorded headcounts within each interval.

The 15-minute means were then aggregated into hourly mean headcounts. Only hours containing all four 15-minute intervals were retained in the complete preliminary hourly dataset.

This aggregation is classified as naive because every raw observation is given equal weight despite the irregular sampling intervals. Empty intervals are retained as missing rather than being interpreted as zero occupancy.

The next stage will construct duration-weighted headcount estimates and compare them with the naive aggregation results.

In [ ]:
# ============================================================
# 1. IMPORT PACKAGES
# ============================================================

from pathlib import Path

import pandas as pd


# ============================================================
# 2. DEFINE PROJECT FOLDERS
# ============================================================

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

raw_folder = project_root / "data" / "raw"
processed_folder = project_root / "data" / "processed"

# Ensure the processed-data folder exists.
processed_folder.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root.resolve())
print("Raw folder exists:", raw_folder.exists())
print("Processed folder exists:", processed_folder.exists())


# ============================================================
# 3. DEFINE AND VALIDATE THE OCCUPANCY FILE
# ============================================================

occupancy_path = (
    raw_folder
    / "5occupancySensor_MaytoDec2024_9MRows.csv"
)

print("\nOccupancy file:", occupancy_path.name)
print("Occupancy file exists:", occupancy_path.exists())

if not occupancy_path.exists():
    raise FileNotFoundError(
        f"Occupancy file not found: {occupancy_path}"
    )


# ============================================================
# 4. LOAD THE REQUIRED OCCUPANCY COLUMNS
# ============================================================

occupancy_columns = [
    "deviceid",
    "floorspaceid",
    "occupancystatus",
    "headcount",
    "collecteddate",
]

occupancy_data = pd.read_csv(
    occupancy_path,
    usecols=occupancy_columns,
    dtype={
        "deviceid": "string",
        "floorspaceid": "string",
        "occupancystatus": "string",
    },
    low_memory=False,
)

print("\nOccupancy data shape:", occupancy_data.shape)
display(occupancy_data.head())


# ============================================================
# 5. CLEAN TIMESTAMPS AND HEADCOUNT
# ============================================================

occupancy_data["timestamp_utc"] = pd.to_datetime(
    occupancy_data["collecteddate"],
    errors="coerce",
    utc=True,
)

occupancy_data["timestamp_local"] = (
    occupancy_data["timestamp_utc"]
    .dt.tz_convert("Australia/Melbourne")
)

occupancy_data["headcount"] = pd.to_numeric(
    occupancy_data["headcount"],
    errors="coerce",
)

print(
    "\nInvalid timestamps:",
    occupancy_data["timestamp_local"].isna().sum(),
)

print(
    "Invalid headcount values:",
    occupancy_data["headcount"].isna().sum(),
)

print(
    "Observation period:",
    occupancy_data["timestamp_local"].min(),
    "to",
    occupancy_data["timestamp_local"].max(),
)


# ============================================================
# 6. INSPECT THE HEADCOUNT DISTRIBUTION
# ============================================================

headcount_summary = occupancy_data["headcount"].describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
    ]
)
headcount_summary = headcount_summary.round(2)
display(headcount_summary)

print(
    "Minimum headcount:",
    occupancy_data["headcount"].min(),
)

print(
    "Maximum headcount:",
    occupancy_data["headcount"].max(),
)

print(
    "Negative headcount records:",
    (occupancy_data["headcount"] < 0).sum(),
)


# ============================================================
# 7. REMOVE UNUSABLE RECORDS
# ============================================================

required_fields = [
    "floorspaceid",
    "timestamp_local",
    "headcount",
]

occupancy_clean = (
    occupancy_data
    .dropna(subset=required_fields)
    .copy()
)

print(f"\nOriginal rows: {len(occupancy_data):,}")
print(f"Usable rows: {len(occupancy_clean):,}")

print(
    "Removed rows:",
    f"{len(occupancy_data) - len(occupancy_clean):,}",
)


# ============================================================
# 8. CREATE ANONYMOUS SPACE LABELS
# ============================================================

space_ids = sorted(
    occupancy_clean["floorspaceid"]
    .dropna()
    .unique()
)

print("\nNumber of unique spaces:", len(space_ids))

if len(space_ids) != 5:
    print(
        "Warning: The dataset does not contain exactly "
        "five unique floor-space IDs."
    )

space_labels = [
    f"Space {chr(65 + index)}"
    for index in range(len(space_ids))
]

space_mapping = dict(
    zip(space_ids, space_labels)
)

occupancy_clean["space"] = (
    occupancy_clean["floorspaceid"]
    .map(space_mapping)
)

print("Anonymous space labels:")
print(list(space_mapping.values()))


# ============================================================
# 9. SUMMARISE EACH SPACE
# ============================================================

space_summary = (
    occupancy_clean
    .groupby("space")
    .agg(
        record_count=("headcount", "size"),
        minimum_headcount=("headcount", "min"),
        mean_headcount=("headcount", "mean"),
        median_headcount=("headcount", "median"),
        maximum_headcount=("headcount", "max"),
        first_observation=("timestamp_local", "min"),
        last_observation=("timestamp_local", "max"),
    )
)

display(space_summary)


# ============================================================
# 10. SORT OBSERVATIONS CHRONOLOGICALLY
# ============================================================

occupancy_clean = (
    occupancy_clean
    .sort_values(
        ["space", "timestamp_local"]
    )
    .reset_index(drop=True)
)

time_order_check = (
    occupancy_clean
    .groupby("space")["timestamp_local"]
    .apply(
        lambda timestamps:
        timestamps.is_monotonic_increasing
    )
)

print("\nChronological ordering check:")
display(time_order_check)


# ============================================================
# 11. CALCULATE SAMPLING INTERVALS
# ============================================================

occupancy_clean["time_since_previous"] = (
    occupancy_clean
    .groupby("space")["timestamp_local"]
    .diff()
)

occupancy_clean["interval_seconds"] = (
    occupancy_clean["time_since_previous"]
    .dt.total_seconds()
)

interval_summary = (
    occupancy_clean
    .groupby("space")["interval_seconds"]
    .agg(
        interval_count="count",
        minimum_seconds="min",
        median_seconds="median",
        mean_seconds="mean",
        maximum_seconds="max",
    )
)

print("\nSampling-interval summary:")
display(interval_summary)

common_intervals = (
    occupancy_clean["interval_seconds"]
    .round(1)
    .value_counts()
    .head(20)
)

print("Most common sampling intervals in seconds:")
display(common_intervals)


# ============================================================
# 12. CHECK FOR LARGE OBSERVATION GAPS
# ============================================================

gap_summary = (
    occupancy_clean
    .assign(
        gap_over_5_minutes=(
            occupancy_clean["interval_seconds"] > 300
        ),
        gap_over_15_minutes=(
            occupancy_clean["interval_seconds"] > 900
        ),
        gap_over_1_hour=(
            occupancy_clean["interval_seconds"] > 3600
        ),
    )
    .groupby("space")
    .agg(
        gaps_over_5_minutes=(
            "gap_over_5_minutes",
            "sum",
        ),
        gaps_over_15_minutes=(
            "gap_over_15_minutes",
            "sum",
        ),
        gaps_over_1_hour=(
            "gap_over_1_hour",
            "sum",
        ),
        maximum_gap_seconds=(
            "interval_seconds",
            "max",
        ),
    )
)

print("\nLarge-gap summary:")
display(gap_summary)


# ============================================================
# 13. CREATE THE NAIVE 15-MINUTE HEADCOUNT SERIES
# ============================================================

headcount_15min_naive = (
    occupancy_clean
    .set_index("timestamp_local")
    .groupby("space")["headcount"]
    .resample("15min")
    .agg(
        mean_headcount="mean",
        minimum_headcount="min",
        maximum_headcount="max",
        observation_count="count",
    )
    .reset_index()
)

print(
    "\n15-minute dataset shape:",
    headcount_15min_naive.shape,
)

display(headcount_15min_naive.head(20))


# ============================================================
# 14. ASSESS 15-MINUTE DATA COVERAGE
# ============================================================

headcount_15min_naive["has_data"] = (
    headcount_15min_naive[
        "observation_count"
    ] > 0
)

intervals_with_data = (
    headcount_15min_naive["has_data"].sum()
)

intervals_without_data = (
    (~headcount_15min_naive["has_data"]).sum()
)

overall_coverage = (
    headcount_15min_naive["has_data"].mean()
    * 100
)

print(
    "\nIntervals with data:",
    f"{intervals_with_data:,}",
)

print(
    "Intervals without data:",
    f"{intervals_without_data:,}",
)

print(
    "Overall 15-minute coverage:",
    f"{overall_coverage:.2f}%",
)

coverage_by_space = (
    headcount_15min_naive
    .groupby("space")
    .agg(
        total_intervals=(
            "timestamp_local",
            "size",
        ),
        intervals_with_data=(
            "has_data",
            "sum",
        ),
    )
)

coverage_by_space["missing_intervals"] = (
    coverage_by_space["total_intervals"]
    - coverage_by_space["intervals_with_data"]
)

coverage_by_space["coverage_percent"] = (
    coverage_by_space["intervals_with_data"]
    / coverage_by_space["total_intervals"]
    * 100
)

print("\nCoverage by space:")
display(coverage_by_space)


# ============================================================
# 15. CREATE PRELIMINARY HOURLY MEAN HEADCOUNT
# ============================================================

hourly_from_15min = (
    headcount_15min_naive
    .set_index("timestamp_local")
    .groupby("space")["mean_headcount"]
    .resample("1h")
    .agg(
        hourly_mean_headcount="mean",
        valid_quarters="count",
    )
    .reset_index()
)

print(
    "\nTotal hourly rows:",
    f"{len(hourly_from_15min):,}",
)

display(hourly_from_15min.head(20))


# ============================================================
# 16. RETAIN HOURS WITH ALL FOUR 15-MINUTE INTERVALS
# ============================================================

hourly_complete = (
    hourly_from_15min.loc[
        hourly_from_15min["valid_quarters"] == 4
    ]
    .copy()
)

complete_hour_count = len(hourly_complete)

incomplete_hour_count = (
    hourly_from_15min["valid_quarters"] < 4
).sum()

print(
    "\nComplete hourly rows:",
    f"{complete_hour_count:,}",
)

print(
    "Incomplete hourly rows:",
    f"{incomplete_hour_count:,}",
)

display(hourly_complete.head(20))


# ============================================================
# 17. VALIDATE THE COMPLETE HOURLY DATASET
# ============================================================

hourly_summary = (
    hourly_complete
    .groupby("space")
    .agg(
        hourly_records=(
            "hourly_mean_headcount",
            "size",
        ),
        first_hour=(
            "timestamp_local",
            "min",
        ),
        last_hour=(
            "timestamp_local",
            "max",
        ),
        minimum_hourly_headcount=(
            "hourly_mean_headcount",
            "min",
        ),
        mean_hourly_headcount=(
            "hourly_mean_headcount",
            "mean",
        ),
        maximum_hourly_headcount=(
            "hourly_mean_headcount",
            "max",
        ),
    )
)

print("\nComplete hourly dataset summary:")
display(hourly_summary)


# ============================================================
# 18. SAVE THE PRELIMINARY PROCESSED OUTPUTS
# ============================================================

headcount_15min_path = (
    processed_folder
    / "headcount_15min_naive.csv"
)

hourly_complete_path = (
    processed_folder
    / "headcount_hourly_from_15min_naive.csv"
)

headcount_15min_naive.to_csv(
    headcount_15min_path,
    index=False,
)

hourly_complete.to_csv(
    hourly_complete_path,
    index=False,
)

print(
    "\nSaved 15-minute dataset to:",
    headcount_15min_path.resolve(),
)

print(
    "Saved hourly dataset to:",
    hourly_complete_path.resolve(),
)

Duration-Weighted Aggregation
The naive aggregation assigns equal weight to every sensor observation. Because the observations occur at irregular intervals, the naive mean may overweight periods in which the sensor records more frequently.

The duration-weighted method treats each observed headcount as remaining active until the next observation for the same space. Each headcount is therefore weighted by the duration for which it was active.

Observation gaps longer than 15 minutes are initially excluded because carrying the previous headcount across an extended period of sensor inactivity may produce unreliable estimates.

In [ ]:
# ============================================================
# DURATION-WEIGHTED AGGREGATION
# ============================================================


# ------------------------------------------------------------
# 1. CALCULATE HOW LONG EACH OBSERVATION REMAINS ACTIVE
# ------------------------------------------------------------

occupancy_clean["next_timestamp_utc"] = (
    occupancy_clean
    .groupby("space")["timestamp_utc"]
    .shift(-1)
)

occupancy_clean["duration_seconds"] = (
    occupancy_clean["next_timestamp_utc"]
    - occupancy_clean["timestamp_utc"]
).dt.total_seconds()

display(
    occupancy_clean[
        [
            "space",
            "timestamp_utc",
            "next_timestamp_utc",
            "headcount",
            "duration_seconds",
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 2. KEEP VALID DURATIONS
# ------------------------------------------------------------

# A headcount observation is carried forward for no more than
# 15 minutes. Longer gaps are treated as missing coverage.

maximum_gap_seconds = 15 * 60

duration_data = (
    occupancy_clean.loc[
        occupancy_clean["duration_seconds"].gt(0)
        & occupancy_clean["duration_seconds"].le(
            maximum_gap_seconds
        ),
        [
            "space",
            "timestamp_utc",
            "next_timestamp_utc",
            "headcount",
            "duration_seconds",
        ],
    ]
    .copy()
)

print(
    "Records available for duration weighting:",
    f"{len(duration_data):,}",
)

print(
    "Percentage of cleaned records retained:",
    f"{len(duration_data) / len(occupancy_clean) * 100:.2f}%",
)


# ------------------------------------------------------------
# 3. DEFINE 15-MINUTE UTC INTERVALS
# ------------------------------------------------------------

# UTC is used for rounding and duration calculations because it
# does not have ambiguous daylight-saving timestamps.

duration_data["interval_start_utc"] = (
    duration_data["timestamp_utc"]
    .dt.floor("15min")
)

duration_data["interval_end_utc"] = (
    duration_data["interval_start_utc"]
    + pd.Timedelta(minutes=15)
)


# ------------------------------------------------------------
# 4. CALCULATE CONTRIBUTION TO THE FIRST INTERVAL
# ------------------------------------------------------------

# The first segment ends either at the next observation or at
# the end of the current 15-minute interval.

duration_data["first_segment_end_utc"] = (
    duration_data[
        [
            "next_timestamp_utc",
            "interval_end_utc",
        ]
    ]
    .min(axis=1)
)

duration_data["first_segment_seconds"] = (
    duration_data["first_segment_end_utc"]
    - duration_data["timestamp_utc"]
).dt.total_seconds()

first_segments = (
    duration_data[
        [
            "space",
            "interval_start_utc",
            "headcount",
            "first_segment_seconds",
        ]
    ]
    .rename(
        columns={
            "first_segment_seconds": "segment_seconds"
        }
    )
    .copy()
)

first_segments["weighted_headcount"] = (
    first_segments["headcount"]
    * first_segments["segment_seconds"]
)


# ------------------------------------------------------------
# 5. CALCULATE CONTRIBUTION TO THE SECOND INTERVAL
# ------------------------------------------------------------

# Because durations longer than 15 minutes have already been
# removed, an observation can cross at most one boundary.

crosses_boundary = (
    duration_data["next_timestamp_utc"]
    > duration_data["interval_end_utc"]
)

print(
    "Observations crossing a 15-minute boundary:",
    f"{crosses_boundary.sum():,}",
)

second_segments = (
    duration_data.loc[
        crosses_boundary,
        [
            "space",
            "interval_end_utc",
            "next_timestamp_utc",
            "headcount",
        ],
    ]
    .rename(
        columns={
            "interval_end_utc": "interval_start_utc"
        }
    )
    .copy()
)

second_segments["segment_seconds"] = (
    second_segments["next_timestamp_utc"]
    - second_segments["interval_start_utc"]
).dt.total_seconds()

second_segments["weighted_headcount"] = (
    second_segments["headcount"]
    * second_segments["segment_seconds"]
)

second_segments = second_segments[
    [
        "space",
        "interval_start_utc",
        "headcount",
        "segment_seconds",
        "weighted_headcount",
    ]
]


# ------------------------------------------------------------
# 6. COMBINE ALL DURATION SEGMENTS
# ------------------------------------------------------------

duration_segments = pd.concat(
    [
        first_segments,
        second_segments,
    ],
    ignore_index=True,
)

duration_segments = (
    duration_segments.loc[
        duration_segments["segment_seconds"] > 0
    ]
    .copy()
)

print(
    "Total duration segments:",
    f"{len(duration_segments):,}",
)

print(
    "Minimum segment duration:",
    duration_segments["segment_seconds"].min(),
)

print(
    "Maximum segment duration:",
    duration_segments["segment_seconds"].max(),
)

display(duration_segments.head(20))


# ------------------------------------------------------------
# 7. AGGREGATE INTO DURATION-WEIGHTED 15-MINUTE INTERVALS
# ------------------------------------------------------------

headcount_15min_weighted = (
    duration_segments
    .groupby(
        [
            "space",
            "interval_start_utc",
        ],
        as_index=False,
    )
    .agg(
        weighted_headcount_sum=(
            "weighted_headcount",
            "sum",
        ),
        observed_seconds=(
            "segment_seconds",
            "sum",
        ),
    )
)

headcount_15min_weighted[
    "weighted_mean_headcount"
] = (
    headcount_15min_weighted[
        "weighted_headcount_sum"
    ]
    / headcount_15min_weighted[
        "observed_seconds"
    ]
)


# ------------------------------------------------------------
# 8. CALCULATE TEMPORAL COVERAGE
# ------------------------------------------------------------

seconds_per_15min_interval = 15 * 60

headcount_15min_weighted["coverage_percent"] = (
    headcount_15min_weighted["observed_seconds"]
    / seconds_per_15min_interval
    * 100
).clip(
    lower=0,
    upper=100,
)

# Convert interval labels back to Melbourne time for reporting.
headcount_15min_weighted["interval_start_local"] = (
    headcount_15min_weighted["interval_start_utc"]
    .dt.tz_convert("Australia/Melbourne")
)

print(
    "Weighted 15-minute dataset shape:",
    headcount_15min_weighted.shape,
)

display(
    headcount_15min_weighted[
        [
            "space",
            "interval_start_utc",
            "interval_start_local",
            "weighted_mean_headcount",
            "observed_seconds",
            "coverage_percent",
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 9. VALIDATE THE WEIGHTED RESULTS
# ------------------------------------------------------------

print(
    "Minimum weighted headcount:",
    headcount_15min_weighted[
        "weighted_mean_headcount"
    ].min(),
)

print(
    "Maximum weighted headcount:",
    headcount_15min_weighted[
        "weighted_mean_headcount"
    ].max(),
)

print(
    "Minimum coverage:",
    headcount_15min_weighted[
        "coverage_percent"
    ].min(),
)

print(
    "Maximum coverage:",
    headcount_15min_weighted[
        "coverage_percent"
    ].max(),
)


# ------------------------------------------------------------
# 10. RETAIN INTERVALS WITH SUFFICIENT COVERAGE
# ------------------------------------------------------------

# A 90% threshold requires at least 810 observed seconds
# within each 15-minute interval.

minimum_coverage_percent = 90

headcount_15min_weighted_complete = (
    headcount_15min_weighted.loc[
        headcount_15min_weighted[
            "coverage_percent"
        ] >= minimum_coverage_percent
    ]
    .copy()
)

print(
    "All weighted intervals:",
    f"{len(headcount_15min_weighted):,}",
)

print(
    "Intervals with at least 90% coverage:",
    f"{len(headcount_15min_weighted_complete):,}",
)

print(
    "Intervals excluded for low coverage:",
    f"{len(headcount_15min_weighted) - len(headcount_15min_weighted_complete):,}",
)


# ------------------------------------------------------------
# 11. PREPARE THE NAIVE DATA FOR COMPARISON
# ------------------------------------------------------------

# The naive dataset was originally calculated using local time.
# Convert its interval labels to UTC before merging.

naive_for_comparison = (
    headcount_15min_naive[
        [
            "space",
            "timestamp_local",
            "mean_headcount",
            "observation_count",
        ]
    ]
    .copy()
)

naive_for_comparison["interval_start_utc"] = (
    naive_for_comparison["timestamp_local"]
    .dt.tz_convert("UTC")
)

naive_for_comparison = (
    naive_for_comparison
    .rename(
        columns={
            "mean_headcount": "naive_mean_headcount"
        }
    )
    [
        [
            "space",
            "interval_start_utc",
            "naive_mean_headcount",
            "observation_count",
        ]
    ]
)


# ------------------------------------------------------------
# 12. COMPARE NAIVE AND DURATION-WEIGHTED ESTIMATES
# ------------------------------------------------------------

aggregation_comparison = (
    naive_for_comparison
    .merge(
        headcount_15min_weighted_complete[
            [
                "space",
                "interval_start_utc",
                "interval_start_local",
                "weighted_mean_headcount",
                "observed_seconds",
                "coverage_percent",
            ]
        ],
        on=[
            "space",
            "interval_start_utc",
        ],
        how="inner",
        validate="one_to_one",
    )
)

aggregation_comparison["difference"] = (
    aggregation_comparison["naive_mean_headcount"]
    - aggregation_comparison["weighted_mean_headcount"]
)

aggregation_comparison["absolute_error"] = (
    aggregation_comparison["difference"]
    .abs()
)

aggregation_comparison["squared_error"] = (
    aggregation_comparison["difference"]
    ** 2
)

display(aggregation_comparison.head(20))


# ------------------------------------------------------------
# 13. CALCULATE OVERALL COMPARISON METRICS
# ------------------------------------------------------------

aggregation_mae = (
    aggregation_comparison["absolute_error"]
    .mean()
)

aggregation_rmse = (
    aggregation_comparison["squared_error"]
    .mean()
    ** 0.5
)

aggregation_bias = (
    aggregation_comparison["difference"]
    .mean()
)

print(
    "Naive versus weighted MAE:",
    f"{aggregation_mae:.4f}",
)

print(
    "Naive versus weighted RMSE:",
    f"{aggregation_rmse:.4f}",
)

print(
    "Mean signed difference:",
    f"{aggregation_bias:.4f}",
)


# ------------------------------------------------------------
# 14. CALCULATE COMPARISON METRICS BY SPACE
# ------------------------------------------------------------

comparison_by_space = (
    aggregation_comparison
    .groupby("space")
    .agg(
        compared_intervals=(
            "absolute_error",
            "size",
        ),
        mean_absolute_error=(
            "absolute_error",
            "mean",
        ),
        root_mean_squared_error=(
            "squared_error",
            lambda values:
            values.mean() ** 0.5,
        ),
        mean_signed_difference=(
            "difference",
            "mean",
        ),
        maximum_absolute_error=(
            "absolute_error",
            "max",
        ),
    )
)

display(comparison_by_space)


# ------------------------------------------------------------
# 15. CREATE THE DURATION-WEIGHTED HOURLY DATASET
# ------------------------------------------------------------

# Aggregate weighted sums and observed durations rather than
# taking an unweighted mean of the 15-minute means.

headcount_hourly_weighted = (
    headcount_15min_weighted_complete
    .set_index("interval_start_utc")
    .groupby("space")
    .resample("1h")
    .agg(
        weighted_headcount_sum=(
            "weighted_headcount_sum",
            "sum",
        ),
        observed_seconds=(
            "observed_seconds",
            "sum",
        ),
        valid_quarters=(
            "weighted_mean_headcount",
            "count",
        ),
    )
    .reset_index()
)

headcount_hourly_weighted[
    "hourly_mean_headcount"
] = (
    headcount_hourly_weighted[
        "weighted_headcount_sum"
    ]
    / headcount_hourly_weighted[
        "observed_seconds"
    ]
)

headcount_hourly_weighted["hour_start_local"] = (
    headcount_hourly_weighted["interval_start_utc"]
    .dt.tz_convert("Australia/Melbourne")
)


# ------------------------------------------------------------
# 16. RETAIN HOURS WITH FOUR VALID QUARTERS
# ------------------------------------------------------------

headcount_hourly_weighted_complete = (
    headcount_hourly_weighted.loc[
        headcount_hourly_weighted[
            "valid_quarters"
        ] == 4
    ]
    .copy()
)

print(
    "Complete weighted hourly rows:",
    f"{len(headcount_hourly_weighted_complete):,}",
)

display(
    headcount_hourly_weighted_complete[
        [
            "space",
            "interval_start_utc",
            "hour_start_local",
            "hourly_mean_headcount",
            "observed_seconds",
            "valid_quarters",
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 17. SAVE THE PROCESSED OUTPUTS
# ------------------------------------------------------------

weighted_15min_path = (
    processed_folder
    / "headcount_15min_duration_weighted.csv"
)

weighted_hourly_path = (
    processed_folder
    / "headcount_hourly_duration_weighted.csv"
)

comparison_path = (
    processed_folder
    / "aggregation_method_comparison.csv"
)

headcount_15min_weighted_complete.to_csv(
    weighted_15min_path,
    index=False,
)

headcount_hourly_weighted_complete.to_csv(
    weighted_hourly_path,
    index=False,
)

aggregation_comparison.to_csv(
    comparison_path,
    index=False,
)

print(
    "Saved weighted 15-minute data to:",
    weighted_15min_path.resolve(),
)

print(
    "Saved weighted hourly data to:",
    weighted_hourly_path.resolve(),
)

print(
    "Saved aggregation comparison to:",
    comparison_path.resolve(),
)

## Duration-weighted aggregation findings

*AI was used to help set up a proper system for the headcount observations
Each headcount observation was treated as a piecewise-constant state that
remained active until the next observation for the same space. Observation
gaps longer than 15 minutes were excluded to avoid carrying potentially
outdated headcounts across periods of sensor inactivity.

Observations crossing a 15-minute boundary were divided between the two
affected intervals. Duration-weighted mean headcount was then calculated by
dividing the sum of headcount-duration products by the valid observed
duration.

Intervals with less than 90% temporal coverage were excluded from the
complete duration-weighted dataset. The duration-weighted values were
compared with the naive arithmetic means using MAE, RMSE and mean signed
difference.